# MLP Output Space Analysis

Properties of MLP layer outputs: effective rank, principal directions,
token alignment, position variation, and summary.

## Why This Matters

MLP output space provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_output_space import (
    mlp_output_rank, mlp_output_direction_analysis,
    mlp_output_token_alignment, mlp_output_position_variation,
    mlp_output_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## MLP Output Rank

How many dimensions does the MLP output occupy?

In [ ]:
for layer in range(4):
    result = mlp_output_rank(model, tokens, layer=layer)
    print(f"  Layer {layer}: rank={result['effective_rank']:.2f}, "
          f"dim_90={result['dim_for_90_pct']}, top_sv={result['top_singular_value']:.4f}")

## Principal Output Directions

In [ ]:
for layer in range(4):
    result = mlp_output_direction_analysis(model, tokens, layer=layer, top_k=3)
    print(f"  Layer {layer}:")
    for d in result['directions']:
        bar = '█' * int(d['variance_explained'] * 30)
        print(f"    Dir {d['rank']}: sv={d['singular_value']:.4f}, var={d['variance_explained']:.2%} {bar}")

## Token Alignment

Which vocabulary tokens does the MLP output point toward?

In [ ]:
for layer in range(4):
    result = mlp_output_token_alignment(model, tokens, layer=layer, top_k=3)
    toks = ', '.join(f'tok{t["token_id"]}({t["logit"]:.2f})' for t in result['top_tokens'])
    print(f"  Layer {layer}: range={result['logit_range']:.4f} | {toks}")

## Position Variation

Is the MLP output position-sensitive?

In [ ]:
for layer in range(4):
    result = mlp_output_position_variation(model, tokens, layer=layer)
    flag = ' [POS-SENSITIVE]' if result['is_position_sensitive'] else ''
    print(f"  Layer {layer}: sim={result['mean_position_similarity']:.4f}, "
          f"cv={result['norm_coefficient_of_variation']:.4f}{flag}")

## Output Summary

In [ ]:
result = mlp_output_summary(model, tokens)
for p in result['per_layer']:
    flag = ' [POS]' if p['is_position_sensitive'] else ''
    print(f"  Layer {p['layer']}: rank={p['effective_rank']:.2f}, "
          f"dim90={p['dim_for_90_pct']}, sim={p['position_similarity']:.4f}{flag}")